<img src="../assets/ga-logo.png" style="float: left; margin: 20px; height: 55px">

# Imbalanced Data

---

### About
Understand the challenges of imbalanced data and apply techniques to address class imbalance in data sets.  This notebook uses the [`imbalanced-learn`](https://imbalanced-learn.org/stable/) library and the examples within it extensively.

`imbalanced-learn` library: https://imbalanced-learn.org/stable/

### Learning Objectives
- Describe when imbalanced data is a problem
- Identify methods for handling imbalanced data and appropriately evaluating performance
- Apply the [`imbalanced-learn`](https://imbalanced-learn.org/stable/index.html) library to oversample minority classes
- Apply the [`imbalanced-learn`](https://imbalanced-learn.org/stable/index.html) library to create synthetic minority class observations

### Notebook Guide

- Imports, and install `imbalanced-learn` package, if needed
- Data SetUp
- Model Setup (Logistic Regression)
- Option 1: Oversample the minority class
- Option 2: Create synthetic data points similar to the minority class
    - SMOTE
    - ADASYN
- Option 3: Overweight minority class importance in algorithms
- Option 4: Change the class cutoff threshold

---
### Metrics

- ALWAYS think about your evaluation metrics AND how a null baseline model performs compared to your model. 
- Check out the confusion matrix.
- Consider using balanced accuracy if you care about all outcome classes equally.



# Imports, and install `imbalanced-learn` package if needed

In [ ]:
# % pip install -U imbalanced-learn

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (balanced_accuracy_score, ConfusionMatrixDisplay, 
                            recall_score, precision_score, f1_score)

from sklearn.datasets import fetch_openml
from imblearn.datasets import make_imbalance

We'll use an example data set used within the User Guide in the docs (see [here](https://imbalanced-learn.org/stable/common_pitfalls.html)): the adult census dataset. As with the User Guide, for the sake of simplicity we will only use the numerical features. Also, we will make the dataset more imbalanced.

In [ ]:
# Option 1: fetch the data from only, select only the numerical features, and make the data imbalanced.  If you do it this way, skip to the 
# next section from here (Data Setup)

# X, y = fetch_openml(data_id=1119, as_frame=True, return_X_y=True, parser='auto')
# X = X.select_dtypes(include="number")
# X, y = make_imbalance(X, y, sampling_strategy={">50K": 300}, random_state=1) # a convenience method for making imbalanced data

In [ ]:
# Option 2: read in the data (note: this data has already been made more imbalanced)

claims = pd.read_csv('../data/imbl_data.csv', index_col=0)

In [ ]:
# select only the numerical features


# Data Setup 

In [ ]:
# train test split 
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, random_state=6) 

In [ ]:
# scale the data
ss = StandardScaler()

X_train_ss = ss.fit_transform(X_train)
X_test_ss = ss.transform(X_test)

# Model Setup (Logistic Regression)

In [ ]:
logr = LogisticRegression(max_iter=1_000)
logr.fit(X_train_ss, y_train)

In [ ]:
logr.score(X_test_ss, y_test)

The accuracy looks very high.

## What metrics do we know?

Helper code provided to review common metrics:

In [ ]:
def evaluation(X_test, y_test, preds, model):
    ConfusionMatrixDisplay.from_estimator(model, X_test, y_test, cmap='Blues')
    
    b_acc = balanced_accuracy_score(y_test, preds)
    recall = recall_score(y_test, preds, pos_label = ">50K")
    prec = precision_score(y_test, preds, pos_label = ">50K")
    f1 = f1_score(y_test, preds, pos_label = ">50K")

    print(f'balanced_accuracy: {b_acc}')
    print(f'recall: {recall}')
    print(f'precision: {prec}')
    print(f'f1 score: {f1}')

    return {
        'balanced_accuracy': b_acc,
        'recall': recall,
        'precision': prec,
        'f1_score': f1
    }

In [ ]:
# setup and call the evaluation function defined above


### Review Null Model, using the balanced accuracy score

Notice that we had very high accuracy with the first model but it's misleading due to the imbalanced classes. We can see that with the [balanced accuracy score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.balanced_accuracy_score.html) (also see [here](https://scikit-learn.org/stable/modules/model_evaluation.html#balanced-accuracy-score)), which is an average of the recall (sensitivity) obtained on each class.

$$
\text{Balanced Accuracy} = \frac{1}{N} \sum_{i=1}^{N} \frac{TP_i}{TP_i + FN_i}
$$

Let's correct for this!

---
# Option 1: Oversample the minority class

Use random sampling with replacement. But use `imbalanced-learn`'s pipeline, not just `df.sample()`.

#### Use **`imbalanced-learn`'s** `make_pipeline` with a `GridSearchCV` object

This ensures the test-set data has similar class imbalance (so you don't overstate your results' generalizability to new data). ⭐️

In [ ]:
from imblearn.over_sampling import RandomOverSampler
from imblearn.pipeline import make_pipeline

### Smarter Move: Grid Search with balanced accuracy as your scoring metric

You can change the scoring metric for `GridSearchCV`. It will then pick the model that hasn't the best mean score on that metric.

### TRY IT!

**Risk:** overfitting the training data. Look at the training and test set scores. Good idea to score on balanced accuracy when grid searching. 👍

---
# Option 2: Create synthetic data points similar to the minority class

**SMOTE** (Synthetic Minority Over-sampling Technique) - make some fake data of the minority class. 

Interpolates to create more minority class observations.

![scatterplots with SMOTE and ADASYN](../assets/smote-docs.png) From the docs: https://imbalanced-learn.org/stable/over_sampling.html#naive-random-over-sampling

In [ ]:
from imblearn.over_sampling import SMOTE

In [ ]:
pipe3 = make_pipeline(SMOTE(random_state=34), LogisticRegression(max_iter=1_000))

In [ ]:
pipe3

In [ ]:
gs3 = GridSearchCV(
    pipe3, 
    {'logisticregression__C':[.1,1,10],}, 
    scoring='balanced_accuracy'
)

In [ ]:
gs3.fit(X_train_ss, y_train)

### TRY IT!

Check the performance 

In [ ]:
gs3.best_params_

In [ ]:
preds3 = gs3.predict(X_test_ss)
evaluation(X_test_ss, y_test, preds3, gs3)

In [ ]:
scores.append(evaluation(X_test_ss, y_test, preds3, gs3))

In [ ]:
pd.DataFrame(scores, index=['logr', 'RandomOverSampler', 'SMOTE'])

### Effects
By training on different data you can change the decision function.

<img src='../assets/decision-function.png'>

**Source:** the docs

### ADASYN
#### Like SMOTE, but focus on the hard observations. Make more data points like them.

"ADASYN focuses on generating samples next to the original samples which are wrongly classified using a k-Nearest Neighbors classifier while the basic implementation of SMOTE will not make any distinction between easy and hard samples to be classified using the nearest neighbors rule." - the docs

In [ ]:
# Make your pipeline with ADASYN 


### Handling Categorical Features 
 When the features are categorical or a mix of categorical and continuous you need to use special versions of SMOTE: ⚠️

- Use `SMOTENC` for mixed features (categorical and continuous).
- Use `SMOTEN` when you have all categorical features.

Review the examples in [the docs](https://imbalanced-learn.org/stable/over_sampling.html#smote-variants)

### SMOTE variations

There are several other SMOTE algorithms you can use. The ones you've seen above should be enough for most problems. If you want to try others, see the [docs](https://imbalanced-learn.org/stable/over_sampling.html#smote-variants).

---
# Option 3: Overweight minority class importance in algorithms

Some models - such as logistic regression - allow for the errors from the observations of one class to carry more weight. With logistic regression, the minority class observations could affect the betas' weights more, which should help with predictions of the minority class.

You can set the weights manually or, as the Logistic Regression `scikit-learn` docs for the `class_weight` argument explain: 
    
    "The "balanced" mode uses the values of y to automatically adjust weights inversely proportional to class frequencies in the input data..."


To learn more about this option, check out [this Machine Learning Mastery article](https://machinelearningmastery.com/cost-sensitive-logistic-regression/).

Note that `imbalanced-learn` also has random forests and boosting ensembles that will balance the data set for training. See more [here](https://imbalanced-learn.org/stable/ensemble.html).

___
# Option 4: Change the class cutoff threshold

We could just adjust our probability threshold cutoff for the minority class to get more minority class predictions.

Our classifier needs to be well calibrated - a 40% probability of class 1 should be correct about 40% of the time.

--- 
## Summary

Much classification data is imbalanced. You've seen how to deal with it to improve model performance while not overstating your model's likely generalizability.

### Overall approach

- Carefully consider your metrics. 
- You can change the `GridSearchCV` scoring metric.
- Make a baseline null model.
- With a task like image classification with a neural net, you just need enough minority samples. Data augmentation can be helpful. The imbalance doesn't matter that much.
- Getting more data is often helpful. 😉

---
### Techniques to improve model performance on minority class ⭐️⭐️⭐️
1. Oversample the minority class
1. Create syntethic data like the minority class (e.g. use SMOTE)
1. Overweight the minority class importance in your algorithm
1. Change the cutoff if there's a `predict_proba()` function

### Imbalanced learn

The `imbalanced-learn` library can help you by providing a `make_pipeline` that keeps your test set safe.

You can use `imbalanced-learn` to train on a random oversampling of your minority class observations. It can also help you create synthetic data with SMOTE of some variation or ADASYN. 

Don't forget to use `imbalanced-learn`'s pipeline so that the training data has the observations you need and the test set is left untouched.